[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/probability_statistics/04_discrete_distributions/exercises.ipynb)

# Exercises — Module 04: Discrete Distributions

20 fully solved problems in four tiers: L0 Concept Checks (4), L1 Foundations (6), L2 Applications in AI/ML and Physics (6), L3 Challenge Proofs (4).

Every numeric answer below is recomputed by the code cell that follows it. Theorem and definition
numbers refer to [`first_principles.ipynb`](first_principles.ipynb).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from math import exp, log, pi, comb, lgamma
from scipy.stats import binom, poisson, geom, nbinom, hypergeom, multinomial

plt.rcParams.update({
    "figure.figsize": (7.0, 4.0),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})
rng = np.random.default_rng(0)
np.set_printoptions(precision=4, suppress=True)

print("preamble ready")

preamble ready


## L0 — Concept Checks

### Problem L0.1 — Which Story, Which Law?

**Statement.** For each scenario name the correct family: (a) 20 independent free throws at 70% accuracy, count makes; (b) drawing 5 cards from a deck and counting aces; (c) rolling a die until the first 6; (d) typos on a page of a long book.

**Intuition.** Each family is the unique answer to one generative question, so the family is decided by auditing four clauses: is $n$ fixed, are trials independent, is $p$ constant, are events rare?

**Solution.**

**Step 1 — (a) Binomial(20, 0.7).** Fixed $n$, independent trials, constant $p$: all three clauses of Definition 3.2 hold, so the count of successes is Binomial.

**Step 2 — (b) Hypergeometric, not Binomial.** Cards are drawn *without replacement*, so trials are dependent and the success probability changes after each draw. With 4 aces among 52 cards, Definition 3.3 gives

$$
P(X = k) = \frac{\binom{4}{k}\binom{48}{5-k}}{\binom{52}{5}}.
$$

By Theorem 4.8 the mean is still $nK/N = 5 \cdot \tfrac{4}{52}$ — linearity needs no independence — but the variance is smaller by the finite-population factor $(N-n)/(N-1) = 47/51$.

**Step 3 — (c) Geometric(1/6).** Independent trials, constant $p$, counting *trials until the first success* (Definition 3.4), so $E[T] = 1/p = 6$.

**Step 4 — (d) Poisson($\lambda$).** Enormously many characters, each mistyped with tiny probability, moderate expected count: the rare-event regime of Theorem 4.4.

$$
\boxed{\text{(a) Binomial;\; (b) Hypergeometric;\; (c) Geometric;\; (d) Poisson}}
$$

*Key takeaway:* Pick the family by auditing the generative clauses — fixed trials, independence, constant $p$, rarity — not by the surface appearance of the data.

In [2]:
N_pop, K_pop, n_draw = 52, 4, 5
k = np.arange(0, 5)
pmf_h = hypergeom.pmf(k, N_pop, K_pop, n_draw)
mean_h = float(np.sum(k * pmf_h))
var_h = float(np.sum(k**2 * pmf_h) - mean_h**2)
f = K_pop / N_pop
print(f"(b) E[X] = {mean_h:.6f}   n*K/N = {n_draw*f:.6f}")
print(f"(b) Var(X) = {var_h:.6f}   binomial nf(1-f) = {n_draw*f*(1-f):.6f}"
      f"   fpc = {(N_pop-n_draw)/(N_pop-1):.6f}")
print(f"(c) E[T] for Geometric(1/6) = {geom.mean(1/6):.4f}")
assert np.isclose(mean_h, n_draw * f)
assert np.isclose(var_h, n_draw * f * (1 - f) * (N_pop - n_draw) / (N_pop - 1))
assert var_h < n_draw * f * (1 - f)

(b) E[X] = 0.384615   n*K/N = 0.384615
(b) Var(X) = 0.327184   binomial nf(1-f) = 0.355030   fpc = 0.921569
(c) E[T] for Geometric(1/6) = 6.0000


The Hypergeometric mean matches the Binomial mean exactly while its variance is strictly smaller, by
precisely the finite-population factor $47/51 \approx 0.922$ — the numerical signature of "without
replacement".

### Problem L0.2 — Is Heads "Due"?

**Statement.** A fair coin lands tails five times in a row. A gambler argues heads is now more likely. Using the Geometric law, state precisely what is and is not true about the waiting time for the first head.

**Intuition.** Memorylessness says the conditional law of the *remaining* wait is the original law, so a run of tails carries no information about the next flip.

**Solution.**

**Step 1 — the survival function.** With $T$ the trial index of the first head and $p = 1/2$, Proof 5.6 gives $P(T \gt n) = (1/2)^n$.

**Step 2 — condition on the run.** For every $n \ge 0$,

$$
P(T \gt 5 + n \mid T \gt 5) = \frac{(1/2)^{5+n}}{(1/2)^{5}} = (1/2)^n = P(T \gt n),
$$

which is exactly the memorylessness property characterizing the Geometric in Theorem 4.6.

**Step 3 — separate the two statements.** What *is* true: the unconditional event "five tails in a row" is rare, probability $1/32$. What is *false*: that its occurrence changes the future. The gambler confuses the low prior probability of a long run with a raised posterior probability of ending it.

$$
\boxed{P(T \gt 5+n \mid T \gt 5) = P(T \gt n) = 2^{-n}; \text{ the coin has no memory}}
$$

*Key takeaway:* Geometric memorylessness is the precise statement that the gambler's fallacy is a fallacy — the expected *remaining* wait stays $1/p$ forever.

In [3]:
flips = rng.random((400_000, 12)) < 0.5          # True = heads
first_head = np.argmax(flips, axis=1) + 1
no_head = ~flips.any(axis=1)
first_head[no_head] = 99                          # censored, far beyond 12

tail5 = first_head > 5
emp_cond = float(np.mean(first_head[tail5] > 7))   # P(T > 5+2 | T > 5)
emp_uncond = float(np.mean(first_head > 2))        # P(T > 2)
print(f"P(T>5) = {tail5.mean():.5f}   theory 2^-5 = {2**-5:.5f}")
print(f"P(T>7 | T>5) = {emp_cond:.5f}   P(T>2) = {emp_uncond:.5f}   theory {2**-2:.5f}")
assert abs(emp_cond - 0.25) < 0.02 and abs(emp_uncond - 0.25) < 0.01

P(T>5) = 0.03145   theory 2^-5 = 0.03125
P(T>7 | T>5) = 0.24964   P(T>2) = 0.25102   theory 0.25000


Both the conditional and the unconditional probability sit at $0.25$ within Monte-Carlo error: after
five tails the remaining wait is statistically indistinguishable from a fresh start.

### Problem L0.3 — Diagnosing Overdispersion

**Statement.** A dataset of daily support tickets has sample mean $\bar{x} = 12.0$ and sample variance $s^2 = 41.5$. Is a Poisson model tenable, and what Negative Binomial dispersion does the data imply?

**Intuition.** Poisson pins the variance to the mean (Theorem 4.3), so a single ratio decides the question.

**Solution.**

**Step 1 — the dispersion index.** By Definition 3.10,

$$
\hat{D} = \frac{s^2}{\bar{x}} = \frac{41.5}{12.0} \approx 3.458 \gg 1,
$$

so the data are strongly **overdispersed** and the Poisson story (constant rate, independent arrivals) is rejected.

**Step 2 — match moments to the mixture.** Theorem 4.7 gives $\mathrm{Var} = \mu + \mu^2/r$ for the Gamma-mixed Poisson, so

$$
41.5 = 12 + \frac{144}{r} \implies r = \frac{144}{29.5} \approx 4.881 .
$$

**Step 3 — read the mechanism.** Small $r$ means strong clustering; $r \to \infty$ recovers Poisson. Either the arrival rate varies day to day (weekday effects, incident spikes) or tickets arrive in correlated bursts.

$$
\boxed{\hat{D} \approx 3.458 \Rightarrow \text{reject Poisson}; \quad \hat{r} \approx 4.881}
$$

*Key takeaway:* The variance-to-mean ratio is a one-number model check; exceeding $1$ signals latent rate heterogeneity, not measurement noise.

In [4]:
xbar, s2 = 12.0, 41.5
D_hat = s2 / xbar
r_hat = xbar**2 / (s2 - xbar)
print(f"dispersion index = {D_hat:.4f}")
print(f"implied NB2 size r = {r_hat:.4f}")
p_nb = r_hat / (r_hat + xbar)
print(f"as scipy nbinom(r, p) with p = {p_nb:.4f}: mean = {nbinom.mean(r_hat, p_nb):.4f}, "
      f"var = {nbinom.var(r_hat, p_nb):.4f}")
assert np.isclose(D_hat, 3.4583, atol=1e-4) and np.isclose(r_hat, 4.8814, atol=1e-4)
assert np.isclose(nbinom.var(r_hat, p_nb), s2)

dispersion index = 3.4583
implied NB2 size r = 4.8814
as scipy nbinom(r, p) with p = 0.2892: mean = 12.0000, var = 41.5000


The fitted Negative Binomial reproduces both moments exactly ($\mu = 12$, $\sigma^2 = 41.5$), which is
what moment matching guarantees; the point of the exercise is that $\hat{D} = 3.46$ already falsifies
Poisson before any fitting is done.

### Problem L0.4 — How Many Free Parameters Does a Softmax Have?

**Statement.** A classifier outputs logits $\mathbf{z} \in \mathbb{R}^K$ passed through a softmax. Show that adding a constant $c$ to every logit changes nothing, and state the dimension of the Categorical parameter space.

**Intuition.** The softmax normalizes, so any common factor in the numerator cancels — the map is constant along the all-ones direction.

**Solution.**

**Step 1 — shift invariance.**

$$
\frac{e^{z_j + c}}{\sum_{i=1}^K e^{z_i + c}} = \frac{e^{c}e^{z_j}}{e^{c}\sum_{i=1}^K e^{z_i}} = \frac{e^{z_j}}{\sum_{i=1}^K e^{z_i}} = p_j .
$$

**Step 2 — count dimensions.** The logit-to-probability map is many-to-one along $\mathbf{1}$, so the model carries one non-identifiable degree of freedom. Correspondingly the Categorical parameter space of Definition 3.7 is the simplex

$$
\Delta^{K-1} = \left\{\mathbf{p} \in \mathbb{R}^K : p_j \ge 0, \; \sum_{j=1}^K p_j = 1\right\},
$$

whose dimension is $K - 1$, not $K$.

**Step 3 — two consequences.** The log-sum-exp trick (subtract $\max_i z_i$) is exact rather than approximate, and binary classification needs only one logit because $K = 2$ leaves one free parameter.

$$
\boxed{\operatorname{softmax}(\mathbf{z} + c\mathbf{1}) = \operatorname{softmax}(\mathbf{z}); \qquad \dim \Delta^{K-1} = K - 1}
$$

*Key takeaway:* Normalization removes one dimension — the redundancy is what licenses numerically safe softmax implementations.

In [5]:
def softmax(z):
    z = np.asarray(z, dtype=float)
    z = z - z.max()                     # exact, by shift invariance
    e = np.exp(z)
    return e / e.sum()


z = rng.normal(size=6) * 3
shifted = softmax(z + 800.0)
print("softmax(z)          =", softmax(z))
print("softmax(z + 800)    =", shifted)
print("max |difference|    =", np.max(np.abs(softmax(z) - shifted)))
with np.errstate(over="ignore"):
    naive = np.exp(z + 800.0)
print("naive exp(z + 800) overflows:", not np.all(np.isfinite(naive)))
assert np.allclose(softmax(z), shifted)
assert not np.all(np.isfinite(naive))

softmax(z)          = [0.0905 0.0078 0.036  0.4942 0.3671 0.0044]
softmax(z + 800)    = [0.0905 0.0078 0.036  0.4942 0.3671 0.0044]
max |difference|    = 2.3647750424515834e-14
naive exp(z + 800) overflows: True


The two probability vectors agree to machine precision while the naive exponential of the shifted
logits overflows to `inf` — shift invariance is not a numerical approximation, it is an exact
symmetry of the Categorical parameterization.

## L1 — Foundations

### Problem L1.1 — Binomial Probabilities and Moments

**Statement.** A message is sent over a channel that flips each of $n = 10$ bits independently with probability $p = 0.1$. Compute $P(\text{no errors})$, $P(\text{at most one error})$, the mean and variance of the error count $X$, and compare with the Poisson approximation and its Le Cam error bound.

**Intuition.** Everything follows from Definition 3.2 and Theorem 4.2; with $\lambda = np = 1$ this sits right at the edge of the rare-event regime, so the Poisson shortcut should be good but not perfect.

**Solution.**

**Step 1 — no errors.** $X \sim \text{Binomial}(10, 0.1)$, so

$$
P(X = 0) = \binom{10}{0}(0.1)^0(0.9)^{10} = 0.9^{10} = 0.348678 .
$$

**Step 2 — exactly one, then at most one.**

$$
P(X = 1) = \binom{10}{1}(0.1)(0.9)^9 = 10 \times 0.1 \times 0.387420 = 0.387420,
$$

so $P(X \le 1) = 0.348678 + 0.387420 = 0.736099$.

**Step 3 — moments.** By Theorem 4.2, $E[X] = np = 1.0$ and $\mathrm{Var}(X) = np(1-p) = 0.9$, so $\mathrm{sd}(X) = \sqrt{0.9} \approx 0.949$.

**Step 4 — the Poisson comparison, with the right metric.** $\text{Poisson}(1)$ gives $P(X = 0) = e^{-1} = 0.367879$, a *pointwise* error of $0.019201$. Le Cam (Theorem 4.5) bounds the *total variation* distance by $\sum_i p_i^2 = 10(0.01) = 0.1$; the exact distance is $d_{\mathrm{TV}} = 0.029312$, comfortably inside. Note the metric: $2\sum_i p_i^2 = 0.2$ bounds the $\ell_1$ distance, which is twice $d_{\mathrm{TV}}$ by Definition 3.9 — see Problem L3.4.

$$
\boxed{P(X=0) = 0.348678, \; P(X \le 1) = 0.736099, \; E[X] = 1, \; \mathrm{Var}(X) = 0.9, \; d_{\mathrm{TV}} = 0.029312 \le 0.1}
$$

*Key takeaway:* Binomial computations are direct; the Poisson shortcut needs its error quoted in a named metric, because the $\ell_1$ constant is twice the total-variation constant.

In [6]:
n, p = 10, 0.1
k = np.arange(0, 60)
pmf_b = binom.pmf(k, n, p)
pmf_p = poisson.pmf(k, n * p)
tv = 0.5 * np.sum(np.abs(pmf_b - pmf_p))
print(f"P(X=0)    = {pmf_b[0]:.6f}")
print(f"P(X<=1)   = {pmf_b[:2].sum():.6f}")
print(f"E[X]      = {binom.mean(n, p):.4f}   Var[X] = {binom.var(n, p):.4f}")
print(f"Poisson(1) P(X=0) = {pmf_p[0]:.6f}   pointwise gap = {abs(pmf_b[0]-pmf_p[0]):.6f}")
print(f"exact d_TV = {tv:.6f}   Le Cam TV bound = {n*p**2:.4f}   l1 bound = {2*n*p**2:.4f}")
assert np.isclose(pmf_b[0], 0.348678, atol=1e-6)
assert np.isclose(pmf_b[:2].sum(), 0.736099, atol=1e-6)
assert np.isclose(tv, 0.029312, atol=1e-6) and tv <= n * p**2

P(X=0)    = 0.348678
P(X<=1)   = 0.736099
E[X]      = 1.0000   Var[X] = 0.9000
Poisson(1) P(X=0) = 0.367879   pointwise gap = 0.019201
exact d_TV = 0.029312   Le Cam TV bound = 0.1000   l1 bound = 0.2000


Every boxed number reproduces. The exact total variation distance $0.0293$ is about $29\%$ of the
Le Cam ceiling $0.1$: the bound is honest and not far off, and quoting $0.2$ instead would have been a
statement about $\ell_1$, not about total variation.

### Problem L1.2 — Expected Wait Without Summing a Series

**Statement.** Derive $E[T]$ and $\mathrm{Var}(T)$ for $T \sim \text{Geometric}(p)$ from the PGF $G_T(s) = \dfrac{ps}{1 - (1-p)s}$, and verify the mean by a first-step argument.

**Intuition.** Theorem 4.1(2) converts moments into two derivatives at $s = 1$, which is far easier than summing $\sum_k k(1-p)^{k-1}p$ directly.

**Solution.**

**Step 1 — differentiate.** Write $q = 1 - p$. By the quotient rule,

$$
G_T'(s) = \frac{p\left(1 - qs\right) + ps\,q}{(1 - qs)^2} = \frac{p}{(1 - qs)^2}, \qquad G_T''(s) = \frac{2pq}{(1 - qs)^3}.
$$

**Step 2 — evaluate at $s = 1$,** where $1 - q = p$:

$$
E[T] = G_T'(1) = \frac{p}{p^2} = \frac{1}{p}, \qquad G_T''(1) = \frac{2pq}{p^3} = \frac{2q}{p^2}.
$$

**Step 3 — assemble the variance** with Theorem 4.1(2):

$$
\mathrm{Var}(T) = G_T''(1) + G_T'(1) - G_T'(1)^2 = \frac{2q}{p^2} + \frac{1}{p} - \frac{1}{p^2} = \frac{2q + p - 1}{p^2} = \frac{q}{p^2} = \frac{1-p}{p^2}.
$$

**Step 4 — first-step check.** Conditioning on the first trial, $E[T] = p(1) + q(1 + E[T])$, so $(1-q)\,E[T] = 1$ and $E[T] = 1/p$, agreeing with Step 2 and with Proof 5.6.

$$
\boxed{E[T] = \frac{1}{p}, \qquad \mathrm{Var}(T) = \frac{1-p}{p^2}}
$$

*Key takeaway:* PGFs convert moment computation into two derivatives; memorylessness gives the same answer through a one-line recursion.

In [7]:
import sympy as sp

s, pp, qq = sp.symbols("s p q", positive=True)
G = pp * s / (1 - (1 - pp) * s)
G1 = sp.simplify(sp.diff(G, s).subs(s, 1))
G2 = sp.simplify(sp.diff(G, s, 2).subs(s, 1))
var_sym = sp.simplify(G2 + G1 - G1**2)
print("G'(1)      =", G1)
print("G''(1)     =", G2)
print("Var        =", sp.factor(var_sym))
assert sp.simplify(G1 - 1 / pp) == 0
assert sp.simplify(var_sym - (1 - pp) / pp**2) == 0

for p_val in (0.2, 0.5, 0.8):
    print(f"p = {p_val}:  scipy mean = {geom.mean(p_val):.6f}, var = {geom.var(p_val):.6f}"
          f"   formula = {1/p_val:.6f}, {(1-p_val)/p_val**2:.6f}")
    assert np.isclose(geom.var(p_val), (1 - p_val) / p_val**2)

G'(1)      = 1/p
G''(1)     = 2*(1 - p)/p**2
Var        = -(p - 1)/p**2
p = 0.2:  scipy mean = 5.000000, var = 20.000000   formula = 5.000000, 20.000000
p = 0.5:  scipy mean = 2.000000, var = 2.000000   formula = 2.000000, 2.000000
p = 0.8:  scipy mean = 1.250000, var = 0.312500   formula = 1.250000, 0.312500


SymPy confirms the two derivatives symbolically, and SciPy's Geometric moments match $1/p$ and
$(1-p)/p^2$ at three values of $p$ — the PGF route and the library agree exactly.

### Problem L1.3 — Poisson Arithmetic

**Statement.** Emails arrive at $\lambda = 4$ per hour. Compute $P(\text{exactly } 2 \text{ in an hour})$, $P(\text{at least } 1 \text{ in } 15 \text{ minutes})$, and $P(\text{exactly } 2 \text{ in an hour} \mid \text{at least } 1)$.

**Intuition.** The "constant rate" clause of the Poisson story means the parameter scales with the window length, so a quarter hour is $\text{Poisson}(1)$.

**Solution.**

**Step 1 — exactly two in an hour,** $X \sim \text{Poisson}(4)$ by Definition 3.6:

$$
P(X = 2) = \frac{4^2 e^{-4}}{2!} = 8 e^{-4} = 8 \times 0.0183156 = 0.146525 .
$$

**Step 2 — the quarter hour.** The rate scales with the window, so $Y \sim \text{Poisson}(4 \times 0.25) = \text{Poisson}(1)$ and

$$
P(Y \ge 1) = 1 - P(Y = 0) = 1 - e^{-1} = 0.632121 .
$$

**Step 3 — the conditional.** Since $\{X = 2\} \subset \{X \ge 1\}$,

$$
P(X = 2 \mid X \ge 1) = \frac{P(X = 2)}{1 - P(X = 0)} = \frac{0.146525}{1 - e^{-4}} = \frac{0.146525}{0.981684} = 0.149259 .
$$

$$
\boxed{P(X=2) = 8e^{-4} \approx 0.146525, \quad P(Y \ge 1) = 1 - e^{-1} \approx 0.632121, \quad P(X = 2 \mid X \ge 1) \approx 0.149259}
$$

*Key takeaway:* Poisson rates are additive in the window length — quartering the interval quarters $\lambda$, which is exactly the "constant rate" clause of the story.

In [8]:
lam = 4.0
p2 = poisson.pmf(2, lam)
p_quarter = 1 - poisson.pmf(0, lam * 0.25)
p_cond = p2 / (1 - poisson.pmf(0, lam))
print(f"P(X=2)            = {p2:.6f}   8e^-4 = {8*exp(-4):.6f}")
print(f"P(Y>=1), 15 min   = {p_quarter:.6f}   1-e^-1 = {1-exp(-1):.6f}")
print(f"P(X=2 | X>=1)     = {p_cond:.6f}")
assert np.isclose(p2, 0.146525, atol=1e-6)
assert np.isclose(p_quarter, 0.632121, atol=1e-6)
assert np.isclose(p_cond, 0.149259, atol=1e-6)

P(X=2)            = 0.146525   8e^-4 = 0.146525
P(Y>=1), 15 min   = 0.632121   1-e^-1 = 0.632121
P(X=2 | X>=1)     = 0.149259


All three values match the hand computation to six decimals, including the rate-scaling step that
turns an hourly $\lambda = 4$ into a quarter-hourly $\lambda = 1$.

### Problem L1.4 — Negative Binomial as a Sum of Geometrics

**Statement.** Show that the trial index $T_r$ of the $r$-th success is a sum of $r$ i.i.d. Geometric($p$) variables, and use this to obtain $E[T_r]$ and $\mathrm{Var}(T_r)$ without touching the PMF.

**Intuition.** Memorylessness makes the process restart after every success, so the waiting time decomposes into independent renewal segments.

**Solution.**

**Step 1 — decompose.** Let $G_1$ be the number of trials up to and including the first success, $G_2$ the number of *additional* trials to the second, and so on. Because the trials are i.i.d. and the process restarts fresh after each success (Theorem 4.6), the $G_i$ are i.i.d. $\text{Geometric}(p)$ and

$$
T_r = G_1 + G_2 + \cdots + G_r .
$$

**Step 2 — moments by linearity and independence.**

$$
E[T_r] = r\,E[G_1] = \frac{r}{p}, \qquad \mathrm{Var}(T_r) = r\,\mathrm{Var}(G_1) = \frac{r(1-p)}{p^2}.
$$

**Step 3 — confirm with the PGF.** By Theorem 4.1(3), $G_{T_r}(s) = \left(\dfrac{ps}{1 - (1-p)s}\right)^{r}$, the $r$-th power of the Geometric PGF; expanding by the negative binomial series reproduces Definition 3.4's $P(T_r = k) = \binom{k-1}{r-1}p^r(1-p)^{k-r}$.

$$
\boxed{T_r = \sum_{i=1}^r G_i, \qquad E[T_r] = \frac{r}{p}, \qquad \mathrm{Var}(T_r) = \frac{r(1-p)}{p^2}}
$$

*Key takeaway:* Decomposing a waiting time into independent renewal segments turns a messy PMF into trivial arithmetic — the same trick powers the coupon-collector analysis.

In [9]:
r_, p_ = 4, 0.3
sims = rng.geometric(p_, size=(200_000, r_)).sum(axis=1)
kk = np.arange(r_, 120)
pmf_exact = np.array([comb(int(kv) - 1, r_ - 1) * p_**r_ * (1 - p_) ** (int(kv) - r_) for kv in kk])
emp = np.array([(sims == kv).mean() for kv in kk])
print(f"E[T_r]:   simulated {sims.mean():.4f}   formula {r_/p_:.4f}")
print(f"Var(T_r): simulated {sims.var():.4f}   formula {r_*(1-p_)/p_**2:.4f}")
print(f"max |empirical pmf - Def. 3.4 pmf| = {np.max(np.abs(emp - pmf_exact)):.5f}")
print(f"pmf sums to {pmf_exact.sum():.8f}")
assert abs(sims.mean() - r_ / p_) < 0.05
assert abs(sims.var() - r_ * (1 - p_) / p_**2) < 1.5
assert np.max(np.abs(emp - pmf_exact)) < 5e-3

E[T_r]:   simulated 13.3228   formula 13.3333
Var(T_r): simulated 31.1782   formula 31.1111
max |empirical pmf - Def. 3.4 pmf| = 0.00149
pmf sums to 1.00000000


The empirical law of $\sum_{i=1}^4 G_i$ tracks Definition 3.4's PMF pointwise to within Monte-Carlo
noise, and both moments land on $r/p = 13.33$ and $r(1-p)/p^2 = 31.11$ — the renewal decomposition is
exact, not an approximation.

### Problem L1.5 — When Do Binomials Add?

**Statement.** Let $X \sim \text{Binomial}(m, p)$ and $Y \sim \text{Binomial}(n, p')$ be independent. Determine exactly when $X + Y$ is Binomial, and compute the mean and variance of $X + Y$ in general.

**Intuition.** PGFs turn convolution into multiplication, and a product of two different linear factors cannot be a power of a single one.

**Solution.**

**Step 1 — multiply PGFs** (Theorem 4.1(3)):

$$
G_{X+Y}(s) = (1 - p + ps)^m (1 - p' + p's)^n .
$$

**Step 2 — the equal case.** If $p = p'$ this collapses to $(1 - p + ps)^{m+n}$, the PGF of $\text{Binomial}(m+n,p)$; uniqueness (Theorem 4.1(1)) gives $X + Y \sim \text{Binomial}(m+n, p)$.

**Step 3 — the unequal case.** If $p \ne p'$ the product has two distinct roots in $s$, namely $1 - 1/p$ and $1 - 1/p'$, so it cannot equal $(1 - \pi + \pi s)^N$, which has one root of multiplicity $N$ — this is Proof 5.2's converse. The resulting law is **Poisson-binomial**.

**Step 4 — moments regardless.**

$$
E[X + Y] = mp + np', \qquad \mathrm{Var}(X + Y) = mp(1-p) + np'(1-p').
$$

$$
\boxed{X + Y \sim \text{Binomial}(m+n, p) \iff p = p'; \text{ otherwise Poisson-binomial}}
$$

*Key takeaway:* Closure under addition requires a shared success probability; generating functions make the condition visible in one line.

In [10]:
m1, p1, n2, p2 = 20, 0.2, 30, 0.7
conv = np.convolve(binom.pmf(np.arange(m1 + 1), m1, p1), binom.pmf(np.arange(n2 + 1), n2, p2))
kk = np.arange(len(conv))
mean_s = float(np.sum(kk * conv))
var_s = float(np.sum(kk**2 * conv) - mean_s**2)
print(f"unequal p: mean = {mean_s:.4f} (formula {m1*p1+n2*p2:.4f}), "
      f"var = {var_s:.4f} (formula {m1*p1*(1-p1)+n2*p2*(1-p2):.4f})")
pi_req = 1 - var_s / mean_s
N_req = mean_s / pi_req
print(f"a Binomial match would need pi = {pi_req:.4f}, N = {N_req:.4f} -> N is not an integer")

conv_eq = np.convolve(binom.pmf(np.arange(m1 + 1), m1, p1), binom.pmf(np.arange(n2 + 1), n2, p1))
print(f"equal p: max |conv - Binomial(50, 0.2)| = "
      f"{np.max(np.abs(conv_eq - binom.pmf(np.arange(len(conv_eq)), m1+n2, p1))):.3e}")
assert np.isclose(mean_s, m1 * p1 + n2 * p2) and np.isclose(var_s, m1*p1*(1-p1)+n2*p2*(1-p2))
assert abs(N_req - round(N_req)) > 1e-3
assert np.allclose(conv_eq, binom.pmf(np.arange(len(conv_eq)), m1 + n2, p1))

unequal p: mean = 25.0000 (formula 25.0000), var = 9.5000 (formula 9.5000)
a Binomial match would need pi = 0.6200, N = 40.3226 -> N is not an integer
equal p: max |conv - Binomial(50, 0.2)| = 1.735e-16


With a common $p$ the convolution equals the Binomial$(50, 0.2)$ PMF to machine precision; with
$p \ne p'$ no integer $N$ reproduces the mean-variance pair, so the sum is genuinely outside the
Binomial family.

### Problem L1.6 — Third Moment and Skewness of the Poisson

**Statement.** Use $M_X(t) = \exp(\lambda(e^t - 1))$ to compute $E[X^3]$ and the skewness of $X \sim \text{Poisson}(\lambda)$.

**Intuition.** Taking a logarithm first turns the MGF into something whose every derivative is the same function, which collapses all cumulants to $\lambda$.

**Solution.**

**Step 1 — the cumulant generating function.** By Theorem 4.3,

$$
K_X(t) = \ln M_X(t) = \lambda(e^t - 1).
$$

**Step 2 — read off the cumulants.** Every derivative of $K_X$ equals $\lambda e^t$, so at $t = 0$ **all cumulants equal $\lambda$**: $\kappa_1 = \kappa_2 = \kappa_3 = \lambda$. Hence $E[X] = \lambda$, $\mathrm{Var}(X) = \lambda$, and the third *central* moment is $\mu_3 = \kappa_3 = \lambda$.

**Step 3 — convert to a raw moment** using $\mu_3 = E[X^3] - 3\mu E[X^2] + 2\mu^3$ with $E[X^2] = \lambda + \lambda^2$:

$$
E[X^3] = \lambda + 3\lambda(\lambda + \lambda^2) - 2\lambda^3 = \lambda^3 + 3\lambda^2 + \lambda .
$$

**Step 4 — skewness.**

$$
\gamma_1 = \frac{\mu_3}{\sigma^3} = \frac{\lambda}{\lambda^{3/2}} = \frac{1}{\sqrt{\lambda}} \xrightarrow[\lambda \to \infty]{} 0,
$$

the quantitative statement that a large-$\lambda$ Poisson looks Gaussian.

$$
\boxed{E[X^3] = \lambda^3 + 3\lambda^2 + \lambda, \qquad \gamma_1 = \lambda^{-1/2}}
$$

*Key takeaway:* For the Poisson the cumulant generating function is linear in $e^t$, so all cumulants collapse to $\lambda$ — the fastest possible route to any moment.

In [11]:
t, lam_s = sp.symbols("t lambda", positive=True)
Kt = lam_s * (sp.exp(t) - 1)
kappas = [sp.simplify(sp.diff(Kt, t, i).subs(t, 0)) for i in (1, 2, 3)]
print("cumulants k1, k2, k3 =", kappas)
assert all(sp.simplify(kv - lam_s) == 0 for kv in kappas)

for lv in (0.5, 3.0, 20.0):
    kk = np.arange(0, int(lv + 40 * np.sqrt(lv) + 40))
    pm = poisson.pmf(kk, lv)
    m3 = float(np.sum(kk**3 * pm))
    skew = float(np.sum((kk - lv) ** 3 * pm) / lv**1.5)
    print(f"lambda = {lv:5.1f}:  E[X^3] = {m3:12.5f}  formula = {lv**3+3*lv**2+lv:12.5f}"
          f"   skew = {skew:.6f}  1/sqrt(l) = {1/np.sqrt(lv):.6f}")
    assert np.isclose(m3, lv**3 + 3 * lv**2 + lv)
    assert np.isclose(skew, 1 / np.sqrt(lv))

cumulants k1, k2, k3 = [lambda, lambda, lambda]
lambda =   0.5:  E[X^3] =      1.37500  formula =      1.37500   skew = 1.414214  1/sqrt(l) = 1.414214
lambda =   3.0:  E[X^3] =     57.00000  formula =     57.00000   skew = 0.577350  1/sqrt(l) = 0.577350
lambda =  20.0:  E[X^3] =   9220.00000  formula =   9220.00000   skew = 0.223607  1/sqrt(l) = 0.223607


SymPy shows the first three cumulants are all $\lambda$, and direct summation of the PMF reproduces
$E[X^3] = \lambda^3+3\lambda^2+\lambda$ and $\gamma_1 = \lambda^{-1/2}$ at three very different rates,
including the strongly skewed $\lambda = 0.5$ and the nearly Gaussian $\lambda = 20$.

## L2 — Applications (AI/ML and Physics)

### Problem L2.1 — Cross-Entropy Is Categorical Maximum Likelihood

**Statement.** Given $n$ i.i.d. labels $y_1, \ldots, y_n \in \{1, \ldots, K\}$ from $\text{Categorical}(\mathbf{p})$, derive the MLE of $\mathbf{p}$ under the simplex constraint, and explain why minimizing cross-entropy is the same optimization.

**Intuition.** The log-likelihood only sees the label counts, and maximizing it against a normalization constraint returns the empirical frequencies.

**Solution.**

**Step 1 — write the log-likelihood.** With $n_j = \#\{i : y_i = j\}$, Definition 3.7 gives

$$
\ell(\mathbf{p}) = \sum_{j=1}^K n_j \ln p_j, \qquad \text{subject to } \sum_{j=1}^K p_j = 1.
$$

**Step 2 — Lagrange stationarity.** With $\mathcal{L} = \sum_j n_j \ln p_j - \nu\left(\sum_j p_j - 1\right)$,

$$
\frac{\partial \mathcal{L}}{\partial p_j} = \frac{n_j}{p_j} - \nu = 0 \implies p_j = \frac{n_j}{\nu}.
$$

**Step 3 — impose the constraint.** Summing gives $\nu = \sum_j n_j = n$, hence $\hat{p}_j = n_j/n$: the MLE is the empirical frequency.

**Step 4 — identify the loss.** With the empirical distribution $\hat{q}_j = n_j/n$,

$$
-\frac{1}{n}\ell(\mathbf{p}) = -\sum_{j=1}^K \hat{q}_j \ln p_j = H(\hat{q}, \mathbf{p}) = H(\hat{q}) + D_{\mathrm{KL}}(\hat{q} \Vert \mathbf{p}).
$$

The entropy term is constant in $\mathbf{p}$, so minimizing cross-entropy is exactly maximizing the Categorical log-likelihood, with minimum at $\mathbf{p} = \hat{q}$ where the KL divergence vanishes.

$$
\boxed{\hat{p}_j = \frac{n_j}{n}; \qquad \text{cross-entropy} = \text{Categorical NLL} = H(\hat{q}) + D_{\mathrm{KL}}(\hat{q} \Vert \mathbf{p})}
$$

*Key takeaway:* The default classification loss is not a heuristic — it is the negative log-likelihood of the Categorical law that the softmax head parameterizes.

In [12]:
K_cls, n_obs = 5, 4000
p_true = np.array([0.35, 0.25, 0.2, 0.15, 0.05])
y = rng.choice(K_cls, size=n_obs, p=p_true)
counts = np.bincount(y, minlength=K_cls)
q_hat = counts / n_obs

H = -np.sum(q_hat * np.log(q_hat))
grid = [q_hat, p_true, np.full(K_cls, 1 / K_cls)]
print("MLE p_hat        =", q_hat, "   true p =", p_true)
for cand in grid:
    ce = -np.sum(q_hat * np.log(cand))
    kl = np.sum(q_hat * np.log(q_hat / cand))
    print(f"  candidate {np.round(cand,3)}  cross-entropy = {ce:.6f}  H + KL = {H+kl:.6f}")
    assert np.isclose(ce, H + kl)
best = min(grid, key=lambda c: -np.sum(q_hat * np.log(c)))
assert np.allclose(best, q_hat)
print("minimizer of cross-entropy is the empirical frequency vector:", np.allclose(best, q_hat))

MLE p_hat        = [0.3445 0.255  0.1978 0.1502 0.0525]    true p = [0.35 0.25 0.2  0.15 0.05]
  candidate [0.344 0.255 0.198 0.15  0.052]  cross-entropy = 1.475586  H + KL = 1.475586
  candidate [0.35 0.25 0.2  0.15 0.05]  cross-entropy = 1.475753  H + KL = 1.475753
  candidate [0.2 0.2 0.2 0.2 0.2]  cross-entropy = 1.609438  H + KL = 1.609438
minimizer of cross-entropy is the empirical frequency vector: True


Cross-entropy equals $H(\hat q) + D_{\mathrm{KL}}$ to machine precision for every candidate, and the
empirical frequency vector beats both the true $\mathbf{p}$ and the uniform vector — which is the
correct behaviour: the MLE minimizes the *empirical* loss, not the population loss.

### Problem L2.2 — Detector Efficiency Is Poisson Thinning (physics)

**Statement.** A source emits photons at $\lambda = 1000$ per second; the detector registers each independently with quantum efficiency $q = 0.25$. Let $N$ be the total photon count in one second and $M$ the registered count. Find the law of $M$, its relative fluctuation, and the correlation between $M$ and the missed count $N - M$.

**Intuition.** Independent retention of Poisson events splits the stream into two independent Poisson streams, so losing photons rescales the rate without changing the family.

**Solution.**

**Step 1 — apply thinning.** By Theorem 4.7(1), $M \sim \text{Poisson}(\lambda q) = \text{Poisson}(250)$ and $N - M \sim \text{Poisson}(750)$ **independently** of $M$, so $\mathrm{Corr}(M, N-M) = 0$. (The total $N$ and the detected $M$ *are* correlated; detected and missed are not.)

**Step 2 — relative fluctuation (shot noise).**

$$
\frac{\sigma_M}{E[M]} = \frac{\sqrt{\lambda q}}{\lambda q} = \frac{1}{\sqrt{250}} = 0.0632 = 6.32\% .
$$

**Step 3 — compare with the ideal detector.** At $q = 1$ the relative fluctuation is $1/\sqrt{1000} = 3.16\%$. Reducing efficiency by a factor of $4$ doubles the relative noise, because the signal-to-noise ratio is $\sqrt{\lambda q}$; recovering $3.16\%$ precision at $q = 0.25$ requires integrating four times longer.

$$
\boxed{M \sim \text{Poisson}(250), \quad \sigma_M/E[M] = 6.32\%, \quad \text{SNR} = \sqrt{\lambda q} \approx 15.81, \quad \mathrm{Corr}(M, N - M) = 0}
$$

*Key takeaway:* Thinning preserves the Poisson family, so an inefficient detector sees the same statistics at a reduced rate — all the cost shows up as $\sqrt{q}$ in signal-to-noise.

In [13]:
lam_ph, q_eff, trials = 1000.0, 0.25, 200_000
N_tot = rng.poisson(lam_ph, size=trials)
M_det = rng.binomial(N_tot, q_eff)
missed = N_tot - M_det

print(f"E[M] = {M_det.mean():.3f} (theory {lam_ph*q_eff:.1f}), "
      f"Var[M] = {M_det.var():.3f} (theory {lam_ph*q_eff:.1f})")
print(f"relative fluctuation = {M_det.std()/M_det.mean():.5f}   theory 1/sqrt(250) = {1/np.sqrt(250):.5f}")
print(f"Corr(M, N-M) = {np.corrcoef(M_det, missed)[0,1]:+.5f}   (theory 0)")
print(f"Corr(M, N)   = {np.corrcoef(M_det, N_tot)[0,1]:+.5f}   (theory sqrt(q) = {np.sqrt(q_eff):.4f})")
print(f"SNR = {np.sqrt(lam_ph*q_eff):.4f}")
assert abs(M_det.var() / M_det.mean() - 1) < 0.03
assert abs(np.corrcoef(M_det, missed)[0, 1]) < 0.01
assert abs(np.corrcoef(M_det, N_tot)[0, 1] - np.sqrt(q_eff)) < 0.02

E[M] = 250.024 (theory 250.0), Var[M] = 249.495 (theory 250.0)
relative fluctuation = 0.06318   theory 1/sqrt(250) = 0.06325
Corr(M, N-M) = -0.00085   (theory 0)
Corr(M, N)   = +0.49819   (theory sqrt(q) = 0.5000)
SNR = 15.8114


The simulated detected counts are equidispersed at $\mathrm{Var}/E \approx 1$, the relative
fluctuation matches $1/\sqrt{250} = 6.32\%$, and $\mathrm{Corr}(M, N-M)$ is zero to three decimals
while $\mathrm{Corr}(M, N) = \sqrt{q} = 0.5$ — exactly the split predicted by Theorem 4.7(1).

### Problem L2.3 — Dropout as Bernoulli Noise

**Statement.** Inverted dropout multiplies each activation $a$ by $B/(1-p)$ with $B \sim \text{Bernoulli}(1-p)$. Show the operation is unbiased, compute the injected variance, and evaluate at $p = 0.5$.

**Intuition.** Dropout is multiplicative Bernoulli noise; the $1/(1-p)$ rescaling is chosen precisely to keep the mean fixed, so the whole effect lands in the variance.

**Solution.**

**Step 1 — unbiasedness.** With $\tilde{a} = a B/(1-p)$ and Definition 3.1,

$$
E[\tilde{a}] = \frac{a}{1-p}E[B] = \frac{a(1-p)}{1-p} = a,
$$

so training-time and inference-time activations share the same mean and no test-time rescaling is needed.

**Step 2 — injected variance.** Since $\mathrm{Var}(B) = p(1-p)$,

$$
\mathrm{Var}(\tilde{a}) = \frac{a^2}{(1-p)^2}\,p(1-p) = a^2\,\frac{p}{1-p}.
$$

**Step 3 — evaluate.** At $p = 0.5$ the factor $p/(1-p) = 1$, so $\mathrm{Var}(\tilde a) = a^2$: the injected noise standard deviation equals the activation magnitude, which is aggressive. At $p = 0.1$ the factor is $0.111$, a mild perturbation.

**Step 4 — why it regularizes.** For a linear unit the expected squared loss picks up a term proportional to $\frac{p}{1-p}\sum_i w_i^2 a_i^2$ — an adaptive $L_2$ penalty weighted by activation energy.

$$
\boxed{E[\tilde{a}] = a, \qquad \mathrm{Var}(\tilde{a}) = a^2\frac{p}{1-p}; \qquad p = 0.5 \implies \mathrm{Var}(\tilde a) = a^2}
$$

*Key takeaway:* Dropout strength is exactly the Bernoulli variance ratio $p/(1-p)$, which is why the useful range of $p$ is narrow and grows explosively as $p \to 1$.

In [14]:
a_val, n_mc = 2.5, 400_000
print("   p    E[a~]/a    Var(a~)/a^2   p/(1-p)")
for p_drop in (0.1, 0.5, 0.9):
    B = (rng.random(n_mc) < (1 - p_drop)).astype(float)
    a_tilde = a_val * B / (1 - p_drop)
    print(f" {p_drop:.1f}   {a_tilde.mean()/a_val:8.5f}   {a_tilde.var()/a_val**2:11.5f}   "
          f"{p_drop/(1-p_drop):8.5f}")
    assert abs(a_tilde.mean() / a_val - 1) < 0.02
    assert abs(a_tilde.var() / a_val**2 - p_drop / (1 - p_drop)) < 0.15 * max(1, p_drop/(1-p_drop))

   p    E[a~]/a    Var(a~)/a^2   p/(1-p)
 0.1    0.99991       0.11119    0.11111
 0.5    0.99979       1.00000    1.00000
 0.9    1.00945       9.07551    9.00000


The mean is preserved at every dropout rate, while the variance ratio tracks $p/(1-p)$: $0.111$ at
$p=0.1$, $1$ at $p=0.5$, and $9$ at $p=0.9$ — an eighty-fold swing over a range of $p$ that looks
narrow on the surface.

### Problem L2.4 — Negative Sampling and the Alias Method

**Statement.** A word2vec model draws negative samples from a Categorical over $K = 10^6$ tokens with tempered probabilities $p_j \propto f_j^{3/4}$. Compare the cost of CDF-search sampling with the alias method, and explain the role of the exponent.

**Intuition.** Both methods pay $O(K)$ once; the difference is per draw, where a binary search costs $O(\log K)$ and an alias table costs $O(1)$.

**Solution.**

**Step 1 — the two costs.** Binary search over the CDF costs $O(\log K) \approx 20$ comparisons per sample with $O(K)$ storage. The **alias method** preprocesses the PMF into $K$ buckets, each holding at most two outcomes plus a threshold, in $O(K)$ time; each draw then takes one uniform for the bucket and one for the threshold — $O(1)$, roughly two memory accesses. Over the billions of negative samples in a training run this is a decisive constant-factor win.

**Step 2 — why temper.** Raw frequencies $f_j$ follow a Zipf law, so a handful of stopwords would absorb almost all negative samples and contribute no learning signal.

**Step 3 — what the exponent does.** Since $x \mapsto x^{3/4}$ is concave and increasing, it compresses the dynamic range:

$$
\frac{p_a}{p_b} = \left(\frac{f_a}{f_b}\right)^{3/4} \lt \frac{f_a}{f_b} \quad \text{whenever } f_a \gt f_b .
$$

For a frequency ratio of $10^4$ the sampling ratio drops to $10^3$: rare words are sampled an order of magnitude more often. The exponent $\alpha = 3/4$ interpolates between the empirical law ($\alpha=1$) and the uniform law ($\alpha=0$).

$$
\boxed{\text{alias: } O(K) \text{ setup}, \; O(1) \text{ per draw}; \qquad p_j \propto f_j^{3/4} \text{ flattens the Zipf tail}}
$$

*Key takeaway:* Large-vocabulary Categorical sampling is an algorithmic problem (alias tables) and a modeling problem (tempering) at the same time.

In [15]:
import time

def build_alias(p):
    K = len(p)
    prob, alias = np.zeros(K), np.zeros(K, dtype=np.int64)
    scaled = p * K
    small = [i for i in range(K) if scaled[i] < 1.0]
    large = [i for i in range(K) if scaled[i] >= 1.0]
    while small and large:
        s, l = small.pop(), large.pop()
        prob[s], alias[s] = scaled[s], l
        scaled[l] -= 1.0 - scaled[s]
        (small if scaled[l] < 1.0 else large).append(l)
    for i in small + large:
        prob[i], alias[i] = 1.0, i
    return prob, alias


def alias_sample(prob, alias, u1, u2):
    i = (u1 * len(prob)).astype(np.int64)
    return np.where(u2 < prob[i], i, alias[i])


K_vocab, n_draws = 200_000, 200_000
ranks = np.arange(1, K_vocab + 1)
f = 1.0 / ranks                                   # Zipf frequencies
p_raw = f / f.sum()
p_temp = f**0.75 / (f**0.75).sum()

cdf = np.cumsum(p_temp)
u = rng.random(n_draws)
t0 = time.perf_counter(); idx_cdf = np.searchsorted(cdf, u); t_cdf = time.perf_counter() - t0

prob, alias = build_alias(p_temp.copy())
u1, u2 = rng.random(n_draws), rng.random(n_draws)
t0 = time.perf_counter(); idx_alias = alias_sample(prob, alias, u1, u2); t_alias = time.perf_counter() - t0

print(f"CDF binary search : {t_cdf*1e3:7.2f} ms for {n_draws} draws")
print(f"alias table       : {t_alias*1e3:7.2f} ms for {n_draws} draws   speedup {t_cdf/t_alias:.2f}x")
print(f"alias empirical mass on token 1 = {(idx_alias==0).mean():.5f}   target {p_temp[0]:.5f}")
print(f"top-token share: raw Zipf {p_raw[0]:.5f}  ->  tempered {p_temp[0]:.5f}")
print(f"ratio p_1/p_1000: raw {p_raw[0]/p_raw[999]:.1f}   tempered {p_temp[0]/p_temp[999]:.1f}"
      f"   (= 1000^0.75 = {1000**0.75:.1f})")
assert abs((idx_alias == 0).mean() - p_temp[0]) < 3 * np.sqrt(p_temp[0] / n_draws) + 1e-4
assert np.isclose(p_temp[0] / p_temp[999], 1000**0.75, rtol=1e-6)

CDF binary search :   29.57 ms for 200000 draws
alias table       :    3.92 ms for 200000 draws   speedup 7.54x
alias empirical mass on token 1 = 0.01247   target 0.01232
top-token share: raw Zipf 0.07823  ->  tempered 0.01232
ratio p_1/p_1000: raw 1000.0   tempered 177.8   (= 1000^0.75 = 177.8)


The alias sampler reproduces the target probability for the most frequent token, and tempering
compresses the head-to-tail ratio from $1000$ to $1000^{3/4} = 178$ exactly as the concavity argument
predicts. The cell uses $K = 2\times10^5$ rather than $10^6$ to keep the notebook fast, and the
measured $6.8\times$ speedup is indicative only: both routines here are vectorized NumPy, so the
timing reflects memory traffic more than the asymptotic $O(\log K)$ versus $O(1)$ gap that dominates
in a scalar training loop.

### Problem L2.5 — Poisson vs Negative Binomial Regression for Count Data

**Statement.** Single-cell RNA-seq counts for a gene have mean $\mu = 5$ and variance $30$ across cells. Fit the Negative Binomial dispersion and quantify how badly a Poisson model understates $P(X \ge 20)$.

**Intuition.** Overdispersion is a statement about the tail: two laws can share a mean and differ by five orders of magnitude on a rare event.

**Solution.**

**Step 1 — fit the dispersion.** With the NB2 parameterization $\mathrm{Var}(X) = \mu + \mu^2/r$ from Theorem 4.7,

$$
30 = 5 + \frac{25}{r} \implies r = 1.0 .
$$

$r = 1$ is the Geometric case — extreme overdispersion, typical of transcriptional bursting where a cell is either actively transcribing or silent.

**Step 2 — the Poisson tail.** Under $\text{Poisson}(5)$ the dominant term is

$$
P(X = 20) = \frac{5^{20}e^{-5}}{20!} = 2.641 \times 10^{-7}, \qquad P(X \ge 20) = 3.452 \times 10^{-7} .
$$

**Step 3 — the Negative Binomial tail.** With $r = 1$ this is the failures-convention Geometric of Definition 3.5 with $p = 1/(1+\mu) = 1/6$, whose survival function is $(1-p)^k$:

$$
P(X \ge 20) = \left(\tfrac{5}{6}\right)^{20} = 0.026084 .
$$

**Step 4 — compare.** The ratio is $7.56 \times 10^{4}$. A Poisson likelihood would treat perfectly ordinary cells as astronomically improbable outliers, inflating significance in differential-expression tests — the false-positive mechanism that motivated NB-based tools such as edgeR and DESeq2.

$$
\boxed{\hat{r} = 1; \quad P(X \ge 20) = 3.45\times 10^{-7} \text{ (Poisson) vs } 0.026084 \text{ (NegBinomial)}, \text{ a ratio of } 7.6\times10^{4}}
$$

*Key takeaway:* Overdispersion is a tail problem — the wrong count model is not slightly wrong, it is wrong by orders of magnitude exactly where inference is decided.

In [16]:
mu_g, var_g = 5.0, 30.0
r_fit = mu_g**2 / (var_g - mu_g)
p_fit = r_fit / (r_fit + mu_g)
tail_pois = float(poisson.sf(19, mu_g))
tail_nb = float(nbinom.sf(19, r_fit, p_fit))
print(f"fitted r = {r_fit:.4f}   p = {p_fit:.4f}")
print(f"NB mean = {nbinom.mean(r_fit, p_fit):.4f}   var = {nbinom.var(r_fit, p_fit):.4f}")
print(f"P(X=20) Poisson  = {poisson.pmf(20, mu_g):.6e}")
print(f"P(X>=20) Poisson = {tail_pois:.6e}")
print(f"P(X>=20) NegBin  = {tail_nb:.6f}   (5/6)^20 = {(5/6)**20:.6f}")
print(f"ratio            = {tail_nb/tail_pois:.4g}")
assert np.isclose(r_fit, 1.0) and np.isclose(nbinom.var(r_fit, p_fit), var_g)
assert np.isclose(tail_nb, (5 / 6) ** 20)
assert 7e4 < tail_nb / tail_pois < 8e4

fitted r = 1.0000   p = 0.1667
NB mean = 5.0000   var = 30.0000
P(X=20) Poisson  = 2.641211e-07
P(X>=20) Poisson = 3.452136e-07
P(X>=20) NegBin  = 0.026084   (5/6)^20 = 0.026084
ratio            = 7.556e+04


The moment fit returns exactly $r = 1$, the Negative Binomial survival function agrees with the
closed form $(5/6)^{20}$, and the two tail probabilities differ by a factor of $7.6\times10^4$ —
the boxed claim, recomputed rather than remembered.

### Problem L2.6 — Multinomial Covariance: Canonical-Ensemble Occupancy and Bag-of-Words (physics)

**Statement.** (a) Place $n = 100$ distinguishable classical particles independently among three energy levels $\varepsilon = (0, k_BT, 2k_BT)$ with Boltzmann weights $p_j \propto e^{-\varepsilon_j/k_BT}$. Compute $\mathrm{Var}(N_0)$, $\mathrm{Cov}(N_0, N_1)$ and their correlation, and say what changes in the grand canonical ensemble. (b) Repeat for a document of $n = 100$ tokens with $p_{\text{the}} = 0.05$, $p_{\text{cat}} = 0.01$.

**Intuition.** Fixing the total forces the counts to compete: one more particle in level $0$ is one fewer available for level $1$, so the covariance must be negative.

**Solution.**

**Step 1 — the occupancy law.** Independent placement of $n$ distinguishable particles makes $\mathbf{N} = (N_0, N_1, N_2)$ Multinomial$(n, \mathbf{p})$ (Definition 3.7), with Boltzmann weights normalized by the partition function $Z = \sum_j e^{-\varepsilon_j/k_BT} = 1 + e^{-1} + e^{-2} = 1.503215$, so

$$
\mathbf{p} = (0.665241,\; 0.244728,\; 0.090031).
$$

**Step 2 — marginals and cross-covariance.** By Table 4.1, $N_j \sim \text{Binomial}(n, p_j)$ and $\mathrm{Cov}(N_i, N_j) = -np_ip_j$:

$$
\mathrm{Var}(N_0) = 100(0.665241)(0.334759) = 22.2695, \qquad \mathrm{Var}(N_1) = 18.4836,
$$

$$
\mathrm{Cov}(N_0, N_1) = -100(0.665241)(0.244728) = -16.2803, \qquad \mathrm{Corr} = \frac{-16.2803}{\sqrt{22.2695 \times 18.4836}} = -0.80244 .
$$

**Step 3 — the ensemble reading.** The strong negative correlation is the fixed-$n$ constraint of the **canonical** ensemble, not a physical interaction: the particles are non-interacting. Relax the constraint — let $N \sim \text{Poisson}(\bar n)$ and place particles independently — and Theorem 4.7(1) says the occupancies become **independent** Poissons with $\mathrm{Cov} = 0$. That is the grand canonical ensemble, and Poisson thinning is the exact statement of the equivalence.

**Step 4 — the same formula for text.** With $p_{\text{the}} = 0.05$, $p_{\text{cat}} = 0.01$, $n = 100$:

$$
\mathrm{Var}(N_{\text{the}}) = 4.75, \quad \mathrm{Var}(N_{\text{cat}}) = 0.99, \quad \mathrm{Cov} = -0.05, \quad \mathrm{Corr} = -0.023 .
$$

The effect is tiny because each word claims a small share of the budget, but it is exactly why raw bag-of-words counts are not independent features.

$$
\boxed{\text{canonical: } \mathrm{Cov}(N_0,N_1) = -16.28,\ \mathrm{Corr} = -0.802; \quad \text{grand canonical: } \mathrm{Cov} = 0; \quad \text{text: } \mathrm{Corr} = -0.023}
$$

*Key takeaway:* Multinomial counts are always negatively correlated because the total is fixed; Poissonizing the total is precisely what removes the correlation.

In [17]:
n_part = 100
weights = np.exp(-np.array([0.0, 1.0, 2.0]))       # Boltzmann, energies in units of kT
p_occ = weights / weights.sum()
print("partition function Z =", weights.sum().round(6), "   p =", p_occ.round(6))

var_theory = n_part * p_occ * (1 - p_occ)
cov_theory = -n_part * p_occ[0] * p_occ[1]
corr_theory = cov_theory / np.sqrt(var_theory[0] * var_theory[1])
print(f"theory: Var(N0) = {var_theory[0]:.4f}  Var(N1) = {var_theory[1]:.4f}  "
      f"Cov = {cov_theory:.4f}  Corr = {corr_theory:.5f}")

samples = rng.multinomial(n_part, p_occ, size=200_000)
C = np.cov(samples.T)
print(f"canonical (fixed n):     Var(N0) = {C[0,0]:.4f}  Cov(N0,N1) = {C[0,1]:.4f}  "
      f"Corr = {C[0,1]/np.sqrt(C[0,0]*C[1,1]):.5f}")

N_rand = rng.poisson(n_part, size=200_000)          # grand canonical: total is Poisson
gc = np.array([rng.multinomial(nv, p_occ) for nv in N_rand[:40_000]])
Cg = np.cov(gc.T)
print(f"grand canonical (N ~ Poisson): Var(N0) = {Cg[0,0]:.4f}  Cov(N0,N1) = {Cg[0,1]:+.4f}  "
      f"theory Var = {n_part*p_occ[0]:.4f}, Cov = 0")

# (b) bag of words
n_tok, p_the, p_cat = 100, 0.05, 0.01
v_the, v_cat = n_tok * p_the * (1 - p_the), n_tok * p_cat * (1 - p_cat)
cov_tc = -n_tok * p_the * p_cat
print(f"text: Var(the) = {v_the:.4f}  Var(cat) = {v_cat:.4f}  Cov = {cov_tc:.4f}  "
      f"Corr = {cov_tc/np.sqrt(v_the*v_cat):.5f}")

assert np.isclose(corr_theory, -0.80244, atol=1e-4)
assert abs(C[0, 1] - cov_theory) < 0.5
assert abs(Cg[0, 1]) < 1.0 and abs(Cg[0, 0] - n_part * p_occ[0]) < 2.0
assert np.isclose(cov_tc / np.sqrt(v_the * v_cat), -0.023057, atol=1e-5)

partition function Z = 1.503215    p = [0.6652 0.2447 0.09  ]
theory: Var(N0) = 22.2695  Var(N1) = 18.4836  Cov = -16.2803  Corr = -0.80244
canonical (fixed n):     Var(N0) = 22.2394  Cov(N0,N1) = -16.2731  Corr = -0.80224
grand canonical (N ~ Poisson): Var(N0) = 66.2875  Cov(N0,N1) = -0.0283  theory Var = 66.5241, Cov = 0
text: Var(the) = 4.7500  Var(cat) = 0.9900  Cov = -0.0500  Corr = -0.02306


The simulated canonical ensemble reproduces $\mathrm{Cov}(N_0,N_1) = -16.28$ and
$\mathrm{Corr} = -0.802$, while Poissonizing the particle number drives the covariance to zero and
makes $\mathrm{Var}(N_0) = E[N_0] = np_0$ — equidispersed independent Poissons, exactly the grand
canonical picture. The text example is the same formula with a tiny correlation, because each token
claims only a small share of the fixed budget.

## L3 — Challenge Proofs

### Problem L3.1 — Chernoff Bound for the Binomial Tail

**Statement.** Let $S \sim \text{Binomial}(n, p)$ with $\mu = np$. Prove the multiplicative Chernoff bound

$$
P(S \ge (1+\delta)\mu) \le \left(\frac{e^{\delta}}{(1+\delta)^{1+\delta}}\right)^{\mu}, \qquad \delta \gt 0,
$$

and deduce $P(S \ge (1+\delta)\mu) \le e^{-\mu\delta^2/3}$ for $0 \lt \delta \le 1$.

**Intuition.** Markov's inequality applied to $e^{tS}$ converts the MGF of Theorem 4.3 into a tail bound, and optimizing over $t$ extracts the best exponent.

**Solution.**

**Step 1 — exponential Markov.** For any $t \gt 0$, monotonicity of $x \mapsto e^{tx}$ and Markov's inequality give

$$
P(S \ge a) = P\left(e^{tS} \ge e^{ta}\right) \le \frac{E\left[e^{tS}\right]}{e^{ta}} = \frac{M_S(t)}{e^{ta}} .
$$

**Step 2 — bound the MGF.** With $M_S(t) = \left(1 + p(e^t - 1)\right)^n$ and $1 + x \le e^{x}$,

$$
M_S(t) \le \exp\left(np(e^t - 1)\right) = \exp\left(\mu(e^t - 1)\right),
$$

which is the *Poisson* MGF of Theorem 4.3: the Binomial is sub-Poisson, the structural reason Chernoff bounds for Binomial and Poisson coincide.

**Step 3 — optimize $t$.** Put $a = (1+\delta)\mu$ and minimize $\exp\left(\mu(e^t-1) - t(1+\delta)\mu\right)$ over $t \gt 0$. Setting the derivative of the exponent to zero gives $\mu e^t = (1+\delta)\mu$, so $t^{\star} = \ln(1+\delta) \gt 0$, and

$$
P(S \ge (1+\delta)\mu) \le \exp\left(\mu\delta - \mu(1+\delta)\ln(1+\delta)\right) = \left(\frac{e^{\delta}}{(1+\delta)^{1+\delta}}\right)^{\mu} . \qquad \blacksquare
$$

**Step 4 — the usable form.** Let $h(\delta) = (1+\delta)\ln(1+\delta) - \delta$, so the bound is $e^{-\mu h(\delta)}$. For $0 \lt \delta \le 1$ expand

$$
h(\delta) = \sum_{k \ge 2}\frac{(-1)^k \delta^k}{k(k-1)} = \frac{\delta^2}{2} - \frac{\delta^3}{6} + \frac{\delta^4}{12} - \cdots
$$

The series alternates with terms decreasing in absolute value on $(0,1]$, so truncating after the cubic term gives $h(\delta) \ge \delta^2/2 - \delta^3/6 \ge \delta^2(1/2 - 1/6) = \delta^2/3$. Hence $P(S \ge (1+\delta)\mu) \le e^{-\mu\delta^2/3}$.

$$
\boxed{P(S \ge (1+\delta)\mu) \le \left(\frac{e^{\delta}}{(1+\delta)^{1+\delta}}\right)^{\mu} \le e^{-\mu\delta^2/3}, \quad 0 \lt \delta \le 1}
$$

*Key takeaway:* Exponential Markov plus MGF optimization gives *exponentially* small tail bounds — the engine behind generalization bounds, randomized algorithm analysis, and load-balancing guarantees.

In [18]:
def chernoff(mu, d):
    return (exp(d) / (1 + d) ** (1 + d)) ** mu


print("   n     p   delta      exact tail     Chernoff      e^{-mu d^2/3}")
for n_c, p_c, d_c in [(100, 0.1, 0.5), (100, 0.1, 1.0), (1000, 0.05, 0.3), (500, 0.2, 0.8)]:
    mu_c = n_c * p_c
    thresh = (1 + d_c) * mu_c
    exact = float(binom.sf(np.ceil(thresh) - 1, n_c, p_c))
    cb, sb = chernoff(mu_c, d_c), exp(-mu_c * d_c**2 / 3)
    print(f"{n_c:6d} {p_c:5.2f} {d_c:6.2f}   {exact:.6e}  {cb:.6e}  {sb:.6e}")
    assert exact <= cb <= sb + 1e-15

# h(delta) >= delta^2/3 on (0, 1]
d_grid = np.linspace(1e-6, 1.0, 5000)
h = (1 + d_grid) * np.log1p(d_grid) - d_grid
print(f"\nmin over (0,1] of h(delta) - delta^2/3 = {np.min(h - d_grid**2/3):.3e}  (must be >= 0)")
assert np.min(h - d_grid**2 / 3) >= -1e-15

   n     p   delta      exact tail     Chernoff      e^{-mu d^2/3}
   100  0.10   0.50   7.257297e-02  3.389249e-01  4.345982e-01
   100  0.10   1.00   1.978561e-03  2.100607e-02  3.567399e-02
  1000  0.05   0.30   2.074990e-02  1.282624e-01  2.231302e-01
   500  0.20   0.80   7.641652e-17  6.230281e-12  5.433142e-10

min over (0,1] of h(delta) - delta^2/3 = 1.667e-13  (must be >= 0)


In every case the exact Binomial tail is below the Chernoff bound, which is below the simplified
$e^{-\mu\delta^2/3}$ form — the chain of inequalities holds in the stated direction. The grid check
confirms $h(\delta) \ge \delta^2/3$ on $(0,1]$, so Step 4's truncation of the alternating series is
valid rather than merely plausible.

### Problem L3.2 — The Coupon Collector

**Statement.** A gacha game has $K$ distinct items, each drawn uniformly at random. Let $T$ be the number of draws to collect all $K$. Compute $E[T]$ and $\mathrm{Var}(T)$ exactly, give the asymptotics, and evaluate at $K = 50$.

**Intuition.** The process has natural renewal points — the moments a *new* item first appears — and between them the wait is Geometric, so Problem L1.4's decomposition applies.

**Solution.**

**Step 1 — decompose at the renewal points.** Let $T_i$ be the number of draws made while exactly $i-1$ distinct items are held, $i = 1, \ldots, K$. Each draw is a success (new item) with probability

$$
p_i = \frac{K - (i-1)}{K},
$$

independently of the past, so $T_i \sim \text{Geometric}(p_i)$ and $T = \sum_{i=1}^K T_i$ with the $T_i$ independent.

**Step 2 — mean by linearity.**

$$
E[T] = \sum_{i=1}^{K} \frac{1}{p_i} = \sum_{i=1}^{K}\frac{K}{K-i+1} = K\sum_{j=1}^{K}\frac{1}{j} = K H_K = K\left(\ln K + \gamma\right) + O(1),
$$

with $\gamma \approx 0.5772$.

**Step 3 — variance by independence,** using $\mathrm{Var}(T_i) = (1-p_i)/p_i^2$ from Problem L1.2:

$$
\mathrm{Var}(T) = \sum_{i=1}^{K}\frac{1-p_i}{p_i^2} = \sum_{j=1}^{K}\left(\frac{K^2}{j^2} - \frac{K}{j}\right) = K^2\sum_{j=1}^{K}\frac{1}{j^2} - K H_K .
$$

This is **exact**. Only now take asymptotics: $\sum_{j \le K} j^{-2} \to \pi^2/6$ and $H_K \sim \ln K$, so

$$
\mathrm{Var}(T) = \frac{\pi^2}{6}K^2 - K\ln K + O(K),
$$

and $\pi^2K^2/6$ alone is a strict *upper* bound, never an equality.

**Step 4 — evaluate at $K = 50$.** $H_{50} = 4.499205$ so $E[T] = 224.960$ draws. The exact variance is $50^2(1.625133) - 50(4.499205) = 3837.872$, so $\mathrm{sd}(T) = 61.951$ — not the $\pi K/\sqrt6 = 64.127$ that the asymptotic-as-equality shortcut gives. The last item alone costs $E[T_K] = K = 50$ draws on average, over $20\%$ of the budget.

$$
\boxed{E[T] = K H_K; \quad \mathrm{Var}(T) = K^2\sum_{j\le K}j^{-2} - K H_K; \quad K = 50 \Rightarrow E[T] = 224.960,\ \mathrm{sd}(T) = 61.951}
$$

*Key takeaway:* Waiting-time problems dissolve once you find the renewal points — and a strict upper bound must never be promoted to an asymptotic equality inside a boxed answer.

In [19]:
K = 50
j = np.arange(1, K + 1)
H_K, S2 = float(np.sum(1 / j)), float(np.sum(1 / j**2))
mean_T = K * H_K
var_T = K**2 * S2 - K * H_K
print(f"H_50 = {H_K:.6f}   sum j^-2 = {S2:.6f}")
print(f"E[T]   = {mean_T:.3f}")
print(f"Var(T) = {var_T:.3f}   sd = {np.sqrt(var_T):.3f}")
print(f"asymptotic pi^2 K^2 / 6 = {(pi**2/6)*K**2:.3f}   -> sd = {np.sqrt((pi**2/6))*K:.3f}  (upper bound)")

trials, horizon = 20_000, 1200
draws = rng.integers(0, K, size=(trials, horizon))
sim = np.empty(trials, dtype=np.int64)
for i in range(trials):
    seen = np.zeros(K, dtype=bool)
    n_seen = 0
    for t in range(horizon):
        x = draws[i, t]
        if not seen[x]:
            seen[x] = True
            n_seen += 1
            if n_seen == K:
                sim[i] = t + 1
                break
print(f"Monte Carlo: mean = {sim.mean():.3f}   var = {sim.var():.3f}   sd = {sim.std():.3f}")
assert var_T < (pi**2 / 6) * K**2
assert abs(sim.mean() - mean_T) < 2.0 and abs(sim.std() - np.sqrt(var_T)) < 3.0

H_50 = 4.499205   sum j^-2 = 1.625133
E[T]   = 224.960
Var(T) = 3837.872   sd = 61.951
asymptotic pi^2 K^2 / 6 = 4112.335   -> sd = 64.127  (upper bound)


Monte Carlo: mean = 225.472   var = 3909.742   sd = 62.528


The exact formula gives $\mathrm{sd}(T) = 61.95$ and the Monte-Carlo run agrees within a draw or two,
while the asymptotic $\pi K/\sqrt6 = 64.13$ overshoots by about $3.5\%$ — the $-KH_K$ correction is
small but real, and it is what distinguishes an exact answer from a bound.

### Problem L3.3 — Deriving the Poisson Process From Three Axioms

**Statement.** Let $N(t)$ count events in $[0,t]$ and assume: (i) $N(0) = 0$ with **stationary, independent increments** — the law of $N(t+h) - N(t)$ depends only on $h$, and increments over disjoint intervals are independent; (ii) $P(N(h) = 1) = \lambda h + o(h)$; (iii) $P(N(h) \ge 2) = o(h)$ (orderliness). Prove $N(t) \sim \text{Poisson}(\lambda t)$.

**Intuition.** Independence turns the count into a product across a partition, and orderliness means an infinitesimal window contributes at most one event — together they give a differential equation whose solution is the Poisson PMF.

**Solution.** Write $P_k(t) = P(N(t) = k)$.

**Step 1 — the empty case.** Split $[0, t+h]$ at $t$. Independence factorizes, and *stationarity* is what lets us replace the increment $N(t+h) - N(t)$ by $N(h)$ in law — without axiom (i)'s stationarity clause, axioms (ii)–(iii) would say nothing about a window starting at $t$:

$$
P_0(t+h) = P_0(t)\,P(N(h) = 0) = P_0(t)\left(1 - \lambda h + o(h)\right),
$$

since (ii) and (iii) force $P(N(h) = 0) = 1 - \lambda h + o(h)$. Rearranging and letting $h \to 0$,

$$
\frac{P_0(t+h) - P_0(t)}{h} = -\lambda P_0(t) + o(1) \implies P_0'(t) = -\lambda P_0(t),
$$

and $P_0(0) = 1$ gives $P_0(t) = e^{-\lambda t}$.

**Step 2 — the recursion for $k \ge 1$.** Condition on the number of events in the final window of length $h$; by orderliness only $0$ or $1$ contribute at order $h$:

$$
P_k(t+h) = P_k(t)(1 - \lambda h) + P_{k-1}(t)\lambda h + o(h) \implies P_k'(t) = -\lambda P_k(t) + \lambda P_{k-1}(t), \quad P_k(0) = 0 .
$$

**Step 3 — integrating factor.** Set $Q_k(t) = e^{\lambda t}P_k(t)$. Then $Q_k'(t) = e^{\lambda t}\left(P_k' + \lambda P_k\right) = \lambda Q_{k-1}(t)$, with $Q_0 \equiv 1$. Induction: assuming $Q_{k-1}(s) = (\lambda s)^{k-1}/(k-1)!$,

$$
Q_k(t) = \lambda\int_0^t \frac{(\lambda s)^{k-1}}{(k-1)!}\,ds = \frac{(\lambda t)^k}{k!} \implies P_k(t) = \frac{(\lambda t)^k e^{-\lambda t}}{k!} . \qquad \blacksquare
$$

**Step 4 — the inter-arrival corollary.** The gap to the first event satisfies $P(T_1 \gt t) = P_0(t) = e^{-\lambda t}$, so inter-arrival times are $\text{Exponential}(\lambda)$: the discrete count law and the continuous waiting law are two faces of the same three axioms.

$$
\boxed{P(N(t) = k) = \frac{(\lambda t)^k e^{-\lambda t}}{k!}, \qquad T_{\text{gap}} \sim \text{Exponential}(\lambda)}
$$

*Key takeaway:* The Poisson law is not postulated but *derived* from stationary independent increments, a constant rate, and no simultaneous events — which is exactly what to check before using it.

In [20]:
lam_pp, T_end, n_bins = 3.0, 2.0, 4000        # discretize the axioms directly
h = T_end / n_bins
p_bin = lam_pp * h                             # P(one event in a bin) = lambda h + o(h)
paths = 200_000
counts = rng.binomial(n_bins, p_bin, size=paths)   # independent, stationary increments

kk = np.arange(0, 25)
emp = np.array([(counts == kv).mean() for kv in kk])
theory = poisson.pmf(kk, lam_pp * T_end)
print(f"bin width h = {h:.5f}, P(event in a bin) = {p_bin:.5f}")
print(f"mean = {counts.mean():.4f} (theory {lam_pp*T_end:.4f}), var = {counts.var():.4f}")
print(f"max |empirical pmf - Poisson({lam_pp*T_end:.0f}) pmf| = {np.max(np.abs(emp-theory)):.5f}")
tv_disc = 0.5 * np.sum(np.abs(binom.pmf(np.arange(0, 200), n_bins, p_bin)
                              - poisson.pmf(np.arange(0, 200), lam_pp * T_end)))
print(f"exact d_TV(discretized model, Poisson) = {tv_disc:.3e}   Le Cam bound = {n_bins*p_bin**2:.3e}")
assert np.max(np.abs(emp - theory)) < 5e-3
assert tv_disc <= n_bins * p_bin**2

bin width h = 0.00050, P(event in a bin) = 0.00150
mean = 5.9966 (theory 6.0000), var = 5.9791
max |empirical pmf - Poisson(6) pmf| = 0.00299
exact d_TV(discretized model, Poisson) = 3.560e-04   Le Cam bound = 9.000e-03


Discretizing the axioms into $4000$ independent, identically distributed windows produces exactly a
Binomial, whose exact distance from $\text{Poisson}(6)$ is $3.56\times10^{-4}$ — inside the Le Cam
ceiling $n(\lambda h)^2 = 9\times10^{-3}$ and shrinking like $h$ as the windows are refined. The
differential-equation derivation and the discrete-window construction describe the same limit.

### Problem L3.4 — Le Cam's Theorem and Non-Identical Trials

**Statement.** Let $S = \sum_{i=1}^n B_i$ with independent $B_i \sim \text{Bernoulli}(p_i)$ and $\lambda = \sum_i p_i$. Give the coupling proof of $d_{\mathrm{TV}}(S, \text{Poisson}(\lambda)) \le \sum_i p_i^2$, and apply it to $n = 1000$ trials with $p_i = 0.002$.

**Intuition.** Total variation is the minimal probability that two coupled variables disagree, so building one explicit coupling — Bernoulli against Poisson, coordinate by coordinate — bounds it.

**Solution.**

**Step 1 — the coupling characterization.**

$$
d_{\mathrm{TV}}(X, Y) = \min_{\text{couplings}} P(X \ne Y),
$$

so any explicit joint construction yields an upper bound.

**Step 2 — couple one coordinate maximally.** For each $i$ build $(B_i, \Pi_i)$ on one space with $B_i \sim \text{Bernoulli}(p_i)$ and $\Pi_i \sim \text{Poisson}(p_i)$ maximally coupled, so $P(B_i = \Pi_i) = \sum_{k}\min\left(P(B_i = k), P(\Pi_i = k)\right)$. Writing $p = p_i$: since $e^{-p} \ge 1-p$, the $k=0$ overlap is $\min(1-p, e^{-p}) = 1-p$; the $k=1$ overlap is $\min(p, pe^{-p}) = pe^{-p}$; and $k \ge 2$ contributes nothing because $P(B_i = k) = 0$ there. Hence

$$
P(B_i \ne \Pi_i) = 1 - (1-p) - pe^{-p} = p\left(1 - e^{-p}\right) \le p\cdot p = p^2,
$$

using $1 - e^{-x} \le x$. The statement worth remembering is $d_{\mathrm{TV}}\left(\text{Bernoulli}(p), \text{Poisson}(p)\right) \le p^2$.

**Step 3 — assemble.** Take the coordinate couplings independently and set $\Pi = \sum_i \Pi_i \sim \text{Poisson}(\lambda)$ by Theorem 4.3. If $S \ne \Pi$ then some coordinate disagreed, so by the union bound

$$
d_{\mathrm{TV}}(S, \Pi) \le P(S \ne \Pi) \le \sum_{i=1}^n P(B_i \ne \Pi_i) \le \sum_{i=1}^n p_i^2 . \qquad \blacksquare
$$

The $\ell_1$ form of Theorem 4.5 carries an extra factor $2$ because $\lVert \cdot \rVert_1 = 2d_{\mathrm{TV}}$ (Definition 3.9) — the distinction Problem L1.1 depends on.

**Step 4 — apply it.** With $n = 1000$, $p_i = 0.002$: $\lambda = 2$ and

$$
d_{\mathrm{TV}} \le \sum_i p_i^2 = 1000 \times 4\times10^{-6} = 0.004 .
$$

Every event probability computed under $\text{Poisson}(2)$ is within $0.004$ of the truth — a hard, finite-$n$ guarantee with no limit taken. The bound depends on $n$ only through $\lambda$ and $\max_i p_i$, since $\sum_i p_i^2 \le \lambda \max_i p_i$: **rarity, not sample size, drives Poisson accuracy**.

$$
\boxed{d_{\mathrm{TV}}\left(S, \text{Poisson}(\lambda)\right) \le \sum_{i=1}^n p_i^2 \le \lambda \max_i p_i; \quad \text{here } \le 0.004, \text{ exact } 4.517\times10^{-4}}
$$

*Key takeaway:* Coupling converts a limit theorem into a quantitative error bound, and reveals that the Poisson approximation never needed identically distributed trials in the first place.

In [21]:
# one-coordinate bound: d_TV(Bernoulli(p), Poisson(p)) = p(1 - e^-p) <= p^2
ps = np.array([0.001, 0.01, 0.05, 0.2, 0.5])
kk = np.arange(0, 40)
exact_one = np.array([0.5 * np.sum(np.abs(binom.pmf(kk, 1, pv) - poisson.pmf(kk, pv))) for pv in ps])
print("   p     exact d_TV     p(1-e^-p)        p^2")
for pv, ev in zip(ps, exact_one):
    print(f"{pv:6.3f}   {ev:.6e}   {pv*(1-exp(-pv)):.6e}   {pv**2:.6e}")
    assert ev <= pv * (1 - exp(-pv)) + 1e-15 <= pv**2 + 1e-15

# the identical-trials application
n_l, p_l = 1000, 0.002
lam_l = n_l * p_l
kk = np.arange(0, 100)
tv_exact = 0.5 * np.sum(np.abs(binom.pmf(kk, n_l, p_l) - poisson.pmf(kk, lam_l)))
print(f"\nn = {n_l}, p_i = {p_l}: lambda = {lam_l}")
print(f"exact d_TV = {tv_exact:.6e}   Le Cam bound sum p_i^2 = {n_l*p_l**2:.6e}")

# non-identical trials, where the theorem earns its keep
p_het = rng.uniform(0.0005, 0.004, size=n_l)
pmf_pb = np.array([1.0])
for pv in p_het:                                  # exact Poisson-binomial by convolution
    pmf_pb = np.convolve(pmf_pb, [1 - pv, pv])
lam_h = p_het.sum()
kk2 = np.arange(len(pmf_pb))
tv_h = 0.5 * np.sum(np.abs(pmf_pb - poisson.pmf(kk2, lam_h)))
print(f"heterogeneous p_i: lambda = {lam_h:.4f}  exact d_TV = {tv_h:.6e}   "
      f"bound = {np.sum(p_het**2):.6e}")
assert tv_exact <= n_l * p_l**2 and np.isclose(tv_exact, 4.517e-4, rtol=1e-2)
assert tv_h <= np.sum(p_het**2)

   p     exact d_TV     p(1-e^-p)        p^2
 0.001   9.995002e-07   9.995002e-07   1.000000e-06
 0.010   9.950166e-05   9.950166e-05   1.000000e-04
 0.050   2.438529e-03   2.438529e-03   2.500000e-03
 0.200   3.625385e-02   3.625385e-02   4.000000e-02
 0.500   1.967347e-01   1.967347e-01   2.500000e-01

n = 1000, p_i = 0.002: lambda = 2.0
exact d_TV = 4.517200e-04   Le Cam bound sum p_i^2 = 4.000000e-03
heterogeneous p_i: lambda = 2.2615  exact d_TV = 6.693503e-04   bound = 6.110977e-03


The per-coordinate distance obeys $p(1-e^{-p}) \le p^2$ at every $p$, the identical-trials case comes
in at $4.52\times10^{-4}$ against the ceiling $4\times10^{-3}$, and the heterogeneous case — where no
limit theorem applies at all — is still inside its own $\sum_i p_i^2$. That is the payoff of the
coupling argument: a bound that never needed identically distributed trials.